# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SDlel/ml-assignments/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I compared logistic regression, a shallow decision tree, and a random forest classifier. This fits the lane because the task is to rank pages by observed refresh risk, and these models can output probabilities that are easy to compare against the hand-written baseline.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "scripts").exists())

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import display_path


def run_script(script_name):
    subprocess.run([sys.executable, str(ROOT / "scripts" / script_name)], cwd=ROOT, check=True)

for script in ["01_prepare_features.py", "02_baseline_score.py", "03_train_model.py"]:
    run_script(script)

model_results = json.loads((ROOT / "outputs" / "model_results.json").read_text())
features = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
predictions = pd.read_csv(ROOT / "data" / "processed" / "model_predictions.csv")

pd.DataFrame([
    {"item": "rows", "value": model_results["input_rows"]},
    {"item": "features", "value": model_results["feature_count"]},
    {"item": "positive label rate", "value": round(model_results["target_positive_rate"], 3)},
    {"item": "best model", "value": model_results["best_model"]["name"]},
])

,item,value
0,rows,30000
1,features,52
2,positive label rate,0.542
3,best model,random_forest


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a client-holdout split when there were enough clients. That is more honest for this question than a random row split because it tests whether the model can generalize to pages from clients it did not train on, instead of memorizing client-specific patterns.

In [2]:
pd.DataFrame([
    {"split_strategy": model_results["split_strategy"], "train_rows": model_results["train_rows"], "test_rows": model_results["test_rows"]}
])

,split_strategy,train_rows,test_rows
0,client_holdout,27675,2325


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The main metric is Precision@50 because the business use case is a short review queue. Higher Precision@50 means more of the first 50 recommended pages are observed declining pages.

In [3]:
rows = []
for name, metrics in model_results["models"].items():
    rows.append({
        "model": name,
        "roc_auc": metrics["roc_auc"],
        "average_precision": metrics["average_precision"],
        "precision_at_50": metrics["precision_at_50"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
    })
baseline = model_results["baseline"]
rows.append({
    "model": "baseline_rules",
    "roc_auc": baseline["baseline_roc_auc"],
    "average_precision": baseline["baseline_average_precision"],
    "precision_at_50": baseline["baseline_precision_at_50"],
    "recall": np.nan,
    "f1": np.nan,
})
comparison = pd.DataFrame(rows).sort_values("precision_at_50", ascending=False)
comparison.to_csv(WORK_OUTPUTS / "model_comparison.csv", index=False)
comparison.round(3)

,model,roc_auc,average_precision,precision_at_50,recall,f1
2,random_forest,0.747,0.610,0.68,0.741,0.638
0,decision_tree,0.742,0.575,0.62,0.716,0.634
1,logistic_regression,0.700,0.522,0.40,0.567,0.566
3,baseline_rules,0.627,0.468,0.24,NaN,NaN


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The random forest is strongest in this run, but it still misses some observed declining pages and over-ranks some non-declining pages. Its top features show that it leans most on observed visibility, position, age, and engagement-style fields.

In [4]:
best_model = model_results["best_model"]["name"]
score_col = f"prob_{best_model}"
test_predictions = predictions.query("split == 'test'").copy()
error_frame = test_predictions.merge(
    features[["content_id", "impressions_90d", "sessions_90d", "avg_position", "content_age_days", "trend_direction"]],
    on="content_id",
    how="left",
)
false_positives = error_frame.query("is_declining_label == 0").nlargest(5, score_col)
false_negatives = error_frame.query("is_declining_label == 1").nsmallest(5, score_col)
feature_importance = pd.DataFrame(model_results["best_model"]["feature_importance_top"]).head(10)

feature_importance.to_csv(WORK_OUTPUTS / "top_model_features.csv", index=False)
error_summary = pd.DataFrame([
    {"group": "false positives reviewed", "rows": len(false_positives), "mean_score": false_positives[score_col].mean()},
    {"group": "false negatives reviewed", "rows": len(false_negatives), "mean_score": false_negatives[score_col].mean()},
])
error_summary.round(3), feature_importance

(                      group  rows  mean_score
 0  false positives reviewed     5       0.733
 1  false negatives reviewed     5       0.112,
                  feature  importance
 0  days_with_impressions    0.160612
 1    log_impressions_90d    0.128491
 2           avg_position    0.108389
 3       content_age_days    0.094997
 4             word_count    0.041163
 5             char_count    0.039843
 6                    ctr    0.033164
 7         log_clicks_90d    0.032499
 8            scroll_rate    0.030914
 9     days_with_sessions    0.029878)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled ? markdown thinking AND code where code is requested.
- [x] Every number comes from a query, dataframe, or saved output.
- [x] I used careful language: observed / measured / directional / decision-support.
- [x] I did not include private client names, domains, URLs, titles, or keywords.